In [1]:
# --- Imports and path constants ---
import os
import polars as pl
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

COMBINED_ROOT = "/scratch/combined_datasets"
H5AD_ROOT     = "/scratch/h5ad_outputs"


# wnm_exc_vis_manuscript
data from Matt Malllory

Three CSVs: morphology features (`AxonRawReatureWide`), metadata + MET labels (`FullMorphMetaData_Master`), and brain-region projection matrix (`ProjectionMatrix`).

In [2]:
CSV_DATA_ROOT = "/root/capsule/data"


def read_csv_labels(
    path: str,
    id_col: str,
    label_col: str,
    rename_label: str = "predicted_label",
    strip_swc: bool = False,
    id_is_int: bool = False,
) -> pd.DataFrame:
    """Read a raw CSV into an (id, predicted_label) DataFrame.

    Parameters
    ----------
    path : str         Path to the CSV file.
    id_col : str       Column to use as cell ID.
    label_col : str    Column to use as the predicted label.
    rename_label : str Output column name for the label (default 'predicted_label').
    strip_swc : bool   If True, strip a trailing '.swc' from ID strings.
    id_is_int : bool   If True, cast id to int64.
    """
    df = pd.read_csv(path, usecols=[id_col, label_col])
    df = df.rename(columns={id_col: "id", label_col: rename_label})
    if strip_swc:
        df["id"] = df["id"].str.removesuffix(".swc")
    if id_is_int:
        df["id"] = df["id"].astype("int64")
    return df.reset_index(drop=True)


def read_csv_feats(
    path: str,
    id_col: str,
    drop_cols: tuple | list = (),
    strip_swc: bool = False,
    id_is_int: bool = False,
) -> pd.DataFrame:
    """Read a raw CSV into an (id, *features) DataFrame.

    Parameters
    ----------
    path : str              Path to the CSV file.
    id_col : str            Column to use as cell ID (renamed to 'id').
    drop_cols : sequence    Non-ID columns to drop (label/metadata columns).
    strip_swc : bool        If True, strip a trailing '.swc' from ID strings.
    id_is_int : bool        If True, cast id to int64.
    """
    df = pd.read_csv(path, index_col=False)
    df = df.rename(columns={id_col: "id"})
    # Drop label/metadata columns and any leftover pandas auto-index columns
    to_drop = [c for c in drop_cols if c in df.columns]
    to_drop += [c for c in df.columns if c != "id" and str(c).startswith("Unnamed:")]
    df = df.drop(columns=to_drop)
    if strip_swc:
        df["id"] = df["id"].str.removesuffix(".swc")
    if id_is_int:
        df["id"] = df["id"].astype("int64")
    return df.reset_index(drop=True)


In [3]:
_WNM_ROOT = os.path.join(CSV_DATA_ROOT, "wnm_exc_vis_manuscript")

_wnm_axon_raw   = pd.read_csv(os.path.join(_WNM_ROOT, "AxonRawReatureWide.csv"))
_wnm_meta       = pd.read_csv(os.path.join(_WNM_ROOT, "FullMorphMetaData_Master.csv"))
_wnm_proj       = pd.read_csv(os.path.join(_WNM_ROOT, "ProjectionMatrix_tip_and_branch_roll_up.csv"))

print(f"AxonRawReatureWide        {_wnm_axon_raw.shape}")
display(_wnm_axon_raw.head(3))

print(f"\nFullMorphMetaData_Master  {_wnm_meta.shape}")
display(_wnm_meta.head(3))

print(f"\nProjectionMatrix (first 6 cols shown)  {_wnm_proj.shape}")
display(_wnm_proj.iloc[:3, :7])  # truncate — 221 columns total


AxonRawReatureWide        (345, 52)


,specimen_id,apical_dendrite_bias_x,apical_dendrite_bias_y,apical_dendrite_depth_pc_0,apical_dendrite_depth_pc_1,apical_dendrite_depth_pc_2,apical_dendrite_depth_pc_3,apical_dendrite_depth_pc_4,apical_dendrite_early_branch_path,apical_dendrite_extent_x,...,axon_max_branch_order,axon_max_euclidean_distance,axon_max_path_distance,axon_mean_contraction,axon_num_branches,axon_soma_percentile_x,axon_soma_percentile_y,axon_total_length,soma_aligned_dist_from_pia,soma_surface_area
0,17109_6201-X4328-Y6753_reg,21.311538,2.306327,-446.959647,-77.960333,-116.411504,-89.641313,-76.897975,0.479785,291.778779,...,18.0,782.489366,2295.547989,0.775924,133.0,0.456671,0.291300,32679.601463,755.648634,0.0
1,17109_6301-X4756-Y24516_reg,39.591795,17.363311,-456.398402,-70.969916,-123.361715,-92.973863,-99.533545,0.484623,255.485239,...,11.0,798.003108,1794.220795,0.789746,83.0,0.434223,0.193180,24944.239274,779.803826,0.0
2,17109_6601-X4384-Y7436_reg,110.472066,-7.655321,-447.486796,-103.720271,-125.564660,-117.194281,-51.706742,0.498860,331.019359,...,10.0,722.175168,2298.330681,0.773319,79.0,0.297082,0.511213,20598.298943,727.831495,0.0



FullMorphMetaData_Master  (341, 17)


,Unnamed: 0,predicted_met_type,probability,ccf_soma_location,ccf_soma_location_nolayer,ccf_soma_x,ccf_soma_y,ccf_soma_z,distance_soma_moved_out_of_brain_correction,cre_line,azimuth,altitude,auto_projection_subclass,dend_derived_predicted_subclass,dend_derived_predicted_probability,local_axon_derived_subclass,met_classifier_routing_call
0,182709_6984-X2452-Y12423_reg.swc,L5 ET-2,0.988,VISpm5,VISpm,8899.823,643.262,4324.473,37.416574,Ai82;Ai139_375886-182709,35.212157,3.519493,ET,ET,0.760219,NaN,ET
1,182709_7126-X2913-Y10535_reg.swc,L5 ET-3,0.918,VISp5,VISp,9117.453,1064.075,3550.508,24.494897,Ai82;Ai139_375886-182709,48.447344,-0.296991,ET,ET,0.949430,NaN,ET
2,182724_5937-X3804-Y11955_reg.swc,L5 ET-2,0.724,VISa5,VISa,7168.289,953.689,3959.651,42.426407,Fezf2-CreER;Ai166_405426-182724,41.861469,1.670020,ET,ET,0.928120,NaN,ET



ProjectionMatrix (first 6 cols shown)  (345, 221)


,Unnamed: 0,ipsi_VISam,ipsi_VISp,ipsi_VISpm,ipsi_VISrl,contra_VISpor,ipsi_CP
0,18864_6734-X4899-Y27447_reg.swc,8287.70664,34450.175934,483.223644,5737.760785,0.000000,0.000000
1,191812_7938-X6892-Y25312_reg.swc,0.00000,794.102517,0.000000,0.000000,1045.437572,9243.339922
2,211550_7718-X19461-Y16950_reg.swc,0.00000,6473.751624,0.000000,0.000000,0.000000,0.000000


# EM_exc_mettypes

`EM_RFC_MET_Predictions_FCNormCorrect_And_Uncorrect.csv` — EM EXC cells with RFC-predicted MET types (corrected and uncorrected). This CSV contains only labels (no morphological features).

In [4]:
_EM_EXC_ROOT = os.path.join(CSV_DATA_ROOT, "EM_exc_mettypes")

_em_exc_met = pd.read_csv(os.path.join(
    _EM_EXC_ROOT, "EM_RFC_MET_Predictions_FCNormCorrect_And_Uncorrect.csv"))
print(f"EM_RFC_MET_Predictions  {_em_exc_met.shape}")
display(_em_exc_met.head(3))


EM_RFC_MET_Predictions  (38267, 7)


,Unnamed: 0,id,uncorrected_rfc_predicted_met_type,uncorrected_rfc_probability,corrected_rfc_predicted_met_type,corrected_rfc_probability,labels_agree
0,0,26468,IT-MET-1,0.868,IT-MET-2,0.978,False
1,1,26554,IT-MET-1,1.000,IT-MET-2,0.948,False
2,2,26556,IT-MET-1,0.994,IT-MET-2,0.948,False


# The visp met taxonomy

In [6]:
# --- VISp EXC MET type taxonomy from cluster schema ---
df_visp_met_taxonomy = (
    pl.read_delta(os.path.join(COMBINED_ROOT, "cluster"))
    .filter(pl.col("project_id") == "visp_met_types")
    .sort(["level", "id"])
)

# Leaf-level EXC (Glutamatergic) MET types
df_visp_exc_met = df_visp_met_taxonomy.filter(
    (pl.col("level") == 2) & (pl.col("parent") == "Glutamatergic")
)

print(f"Full visp_met_types taxonomy: {df_visp_met_taxonomy.shape}")
print(f"EXC MET leaf types          : {df_visp_exc_met.shape[0]} types")
display(df_visp_exc_met.select(["id", "parent", "hex_color", "heirachy_category"]))

Full visp_met_types taxonomy: (48, 9)
EXC MET leaf types          : 17 types


id,parent,hex_color,heirachy_category
str,str,str,str
"""L2/3 IT""","""Glutamatergic""","""#7AE6AB""","""cluster"""
"""L4 IT""","""Glutamatergic""","""#00979D""","""cluster"""
"""L4/L5 IT""","""Glutamatergic""","""#00DDC5""","""cluster"""
"""L5 ET-1 Chrna6""","""Glutamatergic""","""#0000FF""","""cluster"""
"""L5 ET-2""","""Glutamatergic""","""#22737F""","""cluster"""
…,…,…,…
"""L6 CT-2""","""Glutamatergic""","""#578EBF""","""cluster"""
"""L6 IT-1""","""Glutamatergic""","""#C2E32C""","""cluster"""
"""L6 IT-2""","""Glutamatergic""","""#96E32C""","""cluster"""


# MET type labels mismatch and mapping

In [7]:
#EM
mm = list(_em_exc_met.corrected_rfc_predicted_met_type.unique())
mm.sort()
mm

['CT-MET-1',
 'IT-MET-1',
 'IT-MET-2',
 'IT-MET-3',
 'IT-MET-4',
 'IT-MET-5',
 'IT-MET-6',
 'IT-MET-7',
 'IT-MET-8',
 'L6b-MET-1',
 'L6b-MET-2',
 'L6b-MET-3',
 'NP-MET-1',
 'PT-MET-1',
 'PT-MET-2',
 'PT-MET-3']

In [10]:
# Nathan wnm
vv = list(df_visp_exc_met['id'].unique())
vv.sort()
vv

['L2/3 IT',
 'L4 IT',
 'L4/L5 IT',
 'L5 ET-1 Chrna6',
 'L5 ET-2',
 'L5 ET-3',
 'L5 IT-1',
 'L5 IT-2',
 'L5 IT-3 Pld5',
 'L5 NP',
 'L5/L6 IT Car3',
 'L6 CT-1',
 'L6 CT-2',
 'L6 IT-1',
 'L6 IT-2',
 'L6 IT-3',
 'L6b']

In [9]:
# Matt wnm
mw = list(_wnm_meta.predicted_met_type.unique())
mw.sort()
mw

['L2/3 IT',
 'L4 IT',
 'L4/L5 IT',
 'L5 ET-1 Chrna6',
 'L5 ET-2',
 'L5 ET-3',
 'L5 IT-2',
 'L5 NP',
 'L5/L6 IT Car3',
 'L6 CT-1',
 'L6 CT-2',
 'L6 IT-1',
 'L6 IT-2',
 'L6 IT-3',
 'L6b']

In [11]:
# claude mapping based on the manuscript descriptive names
met_to_descriptive = {
    # IT types (8)
    'IT-MET-1': 'L2/3 IT',
    'IT-MET-2': 'L4 IT',
    'IT-MET-3': 'L4/L5 IT',
    'IT-MET-4': 'L5 IT-1',
    'IT-MET-5': 'L5 IT-2 Pld5',
    'IT-MET-6': 'L5/L6 IT Car3',
    'IT-MET-7': 'L6 IT-1',
    'IT-MET-8': 'L6 IT-2',
    # PT/ET types (3)
    'PT-MET-1': 'L5 ET-1 Chrna6',
    'PT-MET-2': 'L5 ET-2',
    'PT-MET-3': 'L5 ET-3 Stac',
    # NP type (1)
    'NP-MET-1': 'L5 NP',
    # CT type (1)
    'CT-MET-1': 'L6 CT',
    # L6b types (3)
    'L6b-MET-1': 'L6b-1',
    'L6b-MET-2': 'L6b-2 Ngf',
    'L6b-MET-3': 'L6b-3',
}

# claude mapping to the existing vis names
met_to_second_list = {
    # IT types — high confidence
    'IT-MET-1': 'L2/3 IT',
    'IT-MET-2': 'L4 IT',
    'IT-MET-3': 'L4/L5 IT',
    'IT-MET-4': 'L5 IT-1',
    'IT-MET-5': 'L5 IT-2',
    'IT-MET-6': 'L5 IT-3 Pld5',
    'IT-MET-7': 'L5/L6 IT Car3',
    'IT-MET-8': 'L6 IT-1',        # paper has L6 IT-1 and L6 IT-2;
                                    # second list has L6 IT-1, L6 IT-2, L6 IT-3
                                    # so one L6 IT in the second list has no MET match

    # PT/ET types — high confidence (PT = pyramidal tract = ET = extratelencephalic)
    'PT-MET-1': 'L5 ET-1 Chrna6',
    'PT-MET-2': 'L5 ET-2',
    'PT-MET-3': 'L5 ET-3',

    # NP — trivial
    'NP-MET-1': 'L5 NP',

    # CT — paper has 1, second list has 2
    'CT-MET-1': 'L6 CT-1',        # L6 CT-2 in second list has no MET match

    # L6b — paper has 3 subtypes, second list collapses to 1
    'L6b-MET-1': 'L6b',
    'L6b-MET-2': 'L6b',           # all three merge into single "L6b"
    'L6b-MET-3': 'L6b',
}

# Existing WNM schemas

In [12]:
# Path to the combined delta lake (same as COMBINED_ROOT in yy12/yy14)
import os
import polars as pl

COMBINED_ROOT = "/scratch/combined_datasets"
assert os.path.exists(COMBINED_ROOT), f"Combined root not found: {COMBINED_ROOT}"
print(f"Loading from: {COMBINED_ROOT}")


Loading from: /scratch/combined_datasets


### Dataset (visp_wnm)

In [13]:
df_wnm_dataset = (
    pl.read_delta(os.path.join(COMBINED_ROOT, "dataset"))
    .filter(pl.col("project_id") == "visp_wnm")
)
print(f"WNM datasets: {df_wnm_dataset.shape}")
df_wnm_dataset


WNM datasets: (1, 5)


id,name,publication,modality,project_id
str,str,str,str,str
"""visp_exc_wnm""","""VISp excitatory whole neuron m…","""doi.org/10.1101/2023.11.25.568…","""MORPHOLOGY""","""visp_wnm"""


### DataItem (visp_wnm)

In [14]:
df_wnm_dataitem = (
    pl.read_delta(os.path.join(COMBINED_ROOT, "dataitem"))
    .filter(pl.col("project_id") == "visp_wnm")
)
print(f"WNM dataitems: {df_wnm_dataitem.shape}")
df_wnm_dataitem.head(5)


WNM dataitems: (341, 4)


id,name,neuroglancer_link,project_id
str,str,str,str
"""182709_6984-X2452-Y12423_reg""","""182709_6984-X2452-Y12423_reg""",null,"""visp_wnm"""
"""182709_7126-X2913-Y10535_reg""","""182709_7126-X2913-Y10535_reg""",null,"""visp_wnm"""
"""182724_5937-X3804-Y11955_reg""","""182724_5937-X3804-Y11955_reg""",null,"""visp_wnm"""
"""182724_6175-X3782-Y10859_reg""","""182724_6175-X3782-Y10859_reg""",null,"""visp_wnm"""
"""182724_6354-X4834-Y8105_reg""","""182724_6354-X4834-Y8105_reg""",null,"""visp_wnm"""


### DataItemDataSetAssociation (visp_wnm)

In [15]:
df_wnm_assoc = (
    pl.read_delta(os.path.join(COMBINED_ROOT, "dataitem_dataset_association"))
    .filter(pl.col("project_id") == "visp_wnm")
)
print(f"WNM associations: {df_wnm_assoc.shape}")
print(f"dataset_ids: {df_wnm_assoc['dataset_id'].unique().to_list()}")
df_wnm_assoc.head(5)


WNM associations: (341, 3)
dataset_ids: ['visp_exc_wnm']


dataitem_id,dataset_id,project_id
str,str,str
"""182709_6984-X2452-Y12423_reg""","""visp_exc_wnm""","""visp_wnm"""
"""182709_7126-X2913-Y10535_reg""","""visp_exc_wnm""","""visp_wnm"""
"""182724_5937-X3804-Y11955_reg""","""visp_exc_wnm""","""visp_wnm"""
"""182724_6175-X3782-Y10859_reg""","""visp_exc_wnm""","""visp_wnm"""
"""182724_6354-X4834-Y8105_reg""","""visp_exc_wnm""","""visp_wnm"""


### CellToClusterMapping (visp_exc_wnm)

In [16]:
df_wnm_mapping = (
    pl.read_delta(os.path.join(COMBINED_ROOT, "celltoclustermapping"))
    .filter(pl.col("project_id") == "visp_exc_wnm")
)
print(f"WNM cell-to-cluster mappings: {df_wnm_mapping.shape}")
print(f"mapping_sets: {df_wnm_mapping['mapping_set'].unique().to_list()}")
df_wnm_mapping.head(5)


WNM cell-to-cluster mappings: (1023, 8)
mapping_sets: ['visp_exc_wnm_mettype_mapping']


id,mapping_set,source_cell,target_cluster,score,probability,notes,project_id
str,str,str,str,f64,f64,str,str
"""182709_6984-X2452-Y12423_reg-L…","""visp_exc_wnm_mettype_mapping""","""182709_6984-X2452-Y12423_reg""","""L5 ET-2""",null,0.988,null,"""visp_exc_wnm"""
"""182709_6984-X2452-Y12423_reg-G…","""visp_exc_wnm_mettype_mapping""","""182709_6984-X2452-Y12423_reg""","""Glutamatergic""",null,null,null,"""visp_exc_wnm"""
"""182709_6984-X2452-Y12423_reg-c…","""visp_exc_wnm_mettype_mapping""","""182709_6984-X2452-Y12423_reg""","""cell""",null,null,null,"""visp_exc_wnm"""
"""182709_7126-X2913-Y10535_reg-L…","""visp_exc_wnm_mettype_mapping""","""182709_7126-X2913-Y10535_reg""","""L5 ET-3""",null,0.918,null,"""visp_exc_wnm"""
"""182709_7126-X2913-Y10535_reg-G…","""visp_exc_wnm_mettype_mapping""","""182709_7126-X2913-Y10535_reg""","""Glutamatergic""",null,null,null,"""visp_exc_wnm"""


### Cluster — WNM target clusters

Clusters that WNM cells are mapped to (via `celltoclustermapping.target_cluster`).

In [17]:
wnm_target_clusters = df_wnm_mapping["target_cluster"].unique().to_list()

df_wnm_clusters = (
    pl.read_delta(os.path.join(COMBINED_ROOT, "cluster"))
    .filter(pl.col("id").is_in(wnm_target_clusters))
)
print(f"Clusters referenced by WNM mappings: {df_wnm_clusters.shape}")
df_wnm_clusters


Clusters referenced by WNM mappings: (21, 9)


id,parent,children,level,score,hex_color,heirachy_category,distance_to_parent,project_id
str,str,list[str],i64,f64,str,str,f64,str
"""cell""",null,"[""GABAergic"", ""Glutamatergic"", ""Non-Neuronal""]",0,null,"""#000000""","""major_class""",null,"""tasic_2018_visp_scrnaseq"""
"""Glutamatergic""","""cell""","[""L4"", ""L2/3 IT"", … ""CR""]",1,null,"""#27AAE1""","""class""",null,"""tasic_2018_visp_scrnaseq"""
"""L2/3 IT""","""Glutamatergic""","[""L2/3 IT VISp Agmat"", ""L2/3 IT VISp Adamts2"", ""L2/3 IT VISp Rrad""]",2,null,"""#94D9A1""","""subclass""",null,"""tasic_2018_visp_scrnaseq"""
"""L6b""","""Glutamatergic""","[""L6b P2ry12"", ""L6b VISp Mup5"", … ""L6b Hsd17b2""]",2,null,"""#25596D""","""subclass""",null,"""tasic_2018_visp_scrnaseq"""
"""cell""",null,"[""GABAergic"", ""Glutamatergic""]",0,null,"""#000000""","""major_class""",null,"""visp_met_types"""
…,…,…,…,…,…,…,…,…
"""L5 ET-3""","""Glutamatergic""",null,2,null,"""#29E043""","""cluster""",null,"""visp_met_types"""
"""L5 NP""","""Glutamatergic""",null,2,null,"""#73CA95""","""cluster""",null,"""visp_met_types"""
"""L6 CT-1""","""Glutamatergic""",null,2,null,"""#74CAFF""","""cluster""",null,"""visp_met_types"""


### CellFeatureSet + CellFeatureDefinition

Morphology features for WNM cells live in `cellfeatures/exc_morph_features`.

In [18]:
# Feature set registry
df_featureset = pl.read_delta(os.path.join(COMBINED_ROOT, "cellfeatureset"))
print(f"All cellfeaturesets: {df_featureset.shape}")
df_featureset


All cellfeaturesets: (3, 4)


id,description,feature_definition_ids,extraction_method
str,str,list[str],str
"""csm_cluster_features_umap""","""Umap reduction fo the features…","[""x_umap"", ""y_umap""]","""umap_learn"""
"""csm_cluster_features""","""Cell features used for cluster…","[""nucleus_volume_um"", ""nucleus_area_um"", … ""ego_count_pca2""]","""Aggegated and computed via htt…"
"""minnie65_std_transform_coordin…","""The coordinates of the cell so…","[""x_medial-lateral"", ""y_dorsal-ventral"", ""z_caudal-rostral""]","""Applied standard_transform to …"


In [19]:
# Feature definitions
df_featdef = pl.read_delta(os.path.join(COMBINED_ROOT, "cellfeaturedefinition"))
print(f"All feature definitions: {df_featdef.shape}")
df_featdef.head(5)


All feature definitions: (87, 6)


id,description,unit,data_type,range_min,range_max
str,str,str,str,f64,f64
"""x_umap""","""umap low dimensional embedding…","""ARBITRARY_UNIT""","""<f4""",null,null
"""y_umap""","""umap low dimensional embedding…","""ARBITRARY_UNIT""","""<f4""",null,null
"""nucleus_volume_um""","""Nucleus volume""","""MICRONS_CUBED""","""<f4""",0.0,NaN
"""nucleus_area_um""","""Nucleus surface area""","""MICRONS_SQUARE""","""<f4""",0.0,NaN
"""nuclear_area_to_volume_ratio""","""Nucleus surface area to volume…","""MICRONS_INVERSE""","""<f4""",0.0,NaN


### CellFeatures — exc_morph_features (visp_exc_wnm)

The `cellfeatures/exc_morph_features` table contains morphology features shared by WNM and patchseq EXC cells (51 feature columns).

In [20]:
df_wnm_feats = (
    pl.read_delta(os.path.join(COMBINED_ROOT, "cellfeatures", "exc_morph_features"))
    .filter(pl.col("project_id") == "visp_exc_wnm")
)
print(f"WNM exc morph features: {df_wnm_feats.shape}")
print(f"  {df_wnm_feats.shape[0]} cells x {df_wnm_feats.shape[1] - 3} features (excl. id/project_id/feature_set_id)")
df_wnm_feats.head(3)


WNM exc morph features: (345, 53)
  345 cells x 50 features (excl. id/project_id/feature_set_id)


id,apical_dendrite_bias_x,apical_dendrite_bias_y,apical_dendrite_depth_pc_0,apical_dendrite_depth_pc_1,apical_dendrite_depth_pc_2,apical_dendrite_depth_pc_3,apical_dendrite_early_branch_path,apical_dendrite_emd_with_basal_dendrite,apical_dendrite_extent_x,apical_dendrite_extent_y,apical_dendrite_frac_above_basal_dendrite,apical_dendrite_frac_below_basal_dendrite,apical_dendrite_frac_intersect_basal_dendrite,apical_dendrite_max_branch_order,apical_dendrite_max_euclidean_distance,apical_dendrite_max_path_distance,apical_dendrite_mean_contraction,apical_dendrite_mean_diameter,apical_dendrite_mean_moments_along_max_distance_projection,apical_dendrite_num_branches,apical_dendrite_num_outer_bifurcations,apical_dendrite_soma_percentile_x,apical_dendrite_soma_percentile_y,apical_dendrite_std_moments_along_max_distance_projection,apical_dendrite_total_length,apical_dendrite_total_surface_area,axon_exit_distance,axon_exit_theta,basal_dendrite_bias_x,basal_dendrite_bias_y,basal_dendrite_calculate_number_of_stems,basal_dendrite_extent_x,basal_dendrite_extent_y,basal_dendrite_frac_above_apical_dendrite,basal_dendrite_frac_below_apical_dendrite,basal_dendrite_frac_intersect_apical_dendrite,basal_dendrite_max_branch_order,basal_dendrite_max_euclidean_distance,basal_dendrite_max_path_distance,basal_dendrite_mean_contraction,basal_dendrite_mean_diameter,basal_dendrite_num_branches,basal_dendrite_soma_percentile_x,basal_dendrite_soma_percentile_y,basal_dendrite_stem_exit_down,basal_dendrite_stem_exit_side,basal_dendrite_stem_exit_up,basal_dendrite_total_length,basal_dendrite_total_surface_area,soma_aligned_dist_from_pia,project_id,feature_set_id
str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i32,f32,f32,f32,f32,f32,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i32,f32,f32,f32,f32,f32,i32,f32,f32,f32,f32,i32,f32,f32,f32,f32,f32,f32,f32,f32,str,str
"""17109_6201-X4328-Y6753_reg""",116.96579,51.220062,-127.799232,-4.550537,-26.701468,-0.559329,0.812125,4.50597,92.771118,130.348099,-0.009498,0.000647,1.006432,4,158.903549,188.477524,0.919582,null,0.297373,13,0.017473,-0.001354,0.264862,0.0951,842.199036,null,null,null,73.426865,73.081833,2,159.217651,171.251968,0.086917,0.038531,0.872697,4,153.118744,180.652649,0.924001,null,14,0.132207,0.186758,0.519306,0.49974,0.001988,928.96991,null,755.648621,"""visp_exc_wnm""","""exc_visp_morph_features"""
"""17109_6301-X4756-Y24516_reg""",68.235786,29.822117,-125.915993,-2.846434,-30.251978,1.331613,0.464351,5.155782,133.943726,187.891479,0.012766,0.059046,0.952927,4,254.745224,298.755646,0.94248,null,0.19052,11,0.017473,0.147617,0.65032,0.047786,833.516296,null,null,null,20.734154,26.343302,3,222.126495,136.760544,0.000506,-0.005611,1.006524,4,158.64447,170.588303,0.954847,null,23,0.300198,0.326576,0.346235,0.33305,0.279562,1155.920532,null,779.803833,"""visp_exc_wnm""","""exc_visp_morph_features"""
"""17109_6601-X4384-Y7436_reg""",144.969879,53.307739,-130.399445,-9.06861,-19.042006,-4.029897,0.573624,12.626372,139.756744,162.011673,0.074466,0.000647,0.921814,4,184.617752,229.434418,0.931142,null,0.417797,18,0.759907,0.012232,0.152526,0.170515,987.185913,null,null,null,46.419273,-36.711784,2,134.836761,159.303131,0.000506,0.232868,0.76836,4,218.932449,225.076797,0.928544,null,19,0.33365,0.71487,0.000094,0.49974,0.418349,976.877747,null,727.831482,"""visp_exc_wnm""","""exc_visp_morph_features"""
